# VAD → ASR → Transcript pipeline

This notebook runs the call-audio pipeline in **three separate stages**. You can stop after any stage and resume later.

| Stage | What it does | Output per call |
|-------|----------------|-----------------|
| **1 — VAD segment** | Silero VAD on Agent + Customer channels, export speech WAV chunks | `segments_manifest.jsonl`, `vad.done` |
| **2 — ASR** | POST each segment to `/recognize/` (resumable checkpoint) | `asr_results.jsonl` |
| **3 — Merge** | Sort by time, merge speaker turns | `final_dialogue.json` |

**Stereo layout (UCall):** channel 0 = Agent, channel 1 = Customer.

**Final transcript format (for LLM tuning):**

```json
["Agent: Em có gì không?", "Customer: Dạ không chị ơi"]
```

Run cells top-to-bottom. Toggle `RUN_STAGE_*` flags in the config cell to run only the stage you need.

## 0 — Configuration

Set paths, limits, and which stages to run. Logic lives in the `pipeline/` package; this notebook only orchestrates stages.

- `MAX_CALLS = None` → process **all** files in `data/raw_audio/` (use a small number while testing).
- `SKIP_VAD_IF_DONE = True` → skip calls that already have `vad.done` (safe to rerun Stage 1).
- ASR reads `SPEECH_API_KEY` from `.env` or your shell environment.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from IPython.display import display

from pipeline import PipelineConfig, run_asr_stage, run_merge_stage, run_vad_stage
from pipeline.metadata import collect_metadata_rows
from pipeline.vad_segment import list_raw_audio_files

# --- Run controls (toggle per session) ---
RUN_STAGE_1_VAD = True
RUN_STAGE_2_ASR = True
RUN_STAGE_3_MERGE = True

# None = all calls in data/raw_audio; set e.g. 3 while testing
MAX_CALLS = None

# --- Pipeline config ---
CFG = PipelineConfig.from_repo(
    max_calls=MAX_CALLS,
    resume_from_checkpoint=True,
    retry_failed_segments=True,
    skip_vad_if_done=True,
)

print(f"Raw audio dir: {CFG.raw_audio_dir}")
print(f"Segment output: {CFG.segment_output_dir}")
print(f"API key loaded: {'yes' if CFG.speech_api_key else 'no'}")
print(f"Calls selected: {len(list_raw_audio_files(CFG))}")


Raw audio dir: /Users/megatunger/Github/call-contextual-extractor/data/raw_audio
Segment output: /Users/megatunger/Github/call-contextual-extractor/data/segmented_audio
API key loaded: yes
Calls selected: 12726


## Optional — Inspect input metadata

Quick sanity check on codec, channels, and duration before batch segmentation. Safe to skip when you already know your inputs.


In [6]:
audio_files = list_raw_audio_files(CFG)

if not audio_files:
    print(f"No .mp3/.wav under {CFG.raw_audio_dir} — run 01_PrepareDataCalls first.")
else:
    meta_df = collect_metadata_rows(audio_files)
    print(f"Metadata ({len(audio_files)} file(s)):")
    display(meta_df)


Metadata (10 file(s)):


file,codec_name,sample_rate_hz_probe,channels_probe,bits_per_sample,duration_sec_probe
str,str,i64,i64,null,f64
"""00066ab6-b592-4ba3-9117-4a7e57…","""mp3""",48000,2,null,214.56
"""000e2489-f26b-4b29-b93d-52fcdf…","""mp3""",48000,2,null,28.992
"""000ecfc0-add8-407f-b8ef-c4fac4…","""mp3""",48000,2,null,18.312
"""00104cfa-2ba1-4a57-8d12-fa045d…","""mp3""",48000,2,null,873.576
"""00155d41-265a-4c79-bb85-52ada9…","""mp3""",48000,2,null,12.672
"""002674d5-b463-4c7a-8df2-d48048…","""mp3""",48000,2,null,24.36
"""002aa6cc-d9d6-49cb-bde0-5e7153…","""mp3""",48000,2,null,22.656
"""002c715a-ad48-4861-8f18-df63c2…","""mp3""",48000,2,null,428.256
"""002c7457-6844-43d6-9b17-9f519c…","""mp3""",48000,2,null,180.96


## Stage 1 — VAD segmentation (no API calls)

For each raw call file:

1. Load stereo audio at 16 kHz
2. Run Silero VAD separately on **Agent** and **Customer**
3. Write WAV segments under `data/segmented_audio/<call_id>/agent|customer/`
4. Save `segments_manifest.jsonl` and mark `vad.done`

**Resume:** calls with `vad.done` are skipped when `SKIP_VAD_IF_DONE=True`.

Run this stage alone when you have thousands of files — ASR can run later in Stage 2.


In [2]:
if RUN_STAGE_1_VAD:
    vad_summary = run_vad_stage(CFG)
    print(vad_summary)
else:
    print("Skipped Stage 1 (RUN_STAGE_1_VAD=False)")


VAD calls already done: 0/12726 (0.0%)


Stage 1 · VAD segment calls:   0%|          | 0/12726 [00:00<?, ?call/s]

KeyboardInterrupt: 

## Stage 2 — ASR recognition (segment → API)

Reads `segments_manifest.jsonl` for every segmented call and POSTs each WAV to `/recognize/`.

**Resume:** successful rows in `asr_results.jsonl` are skipped; failed rows are retried when `retry_failed_segments=True`.

Progress bar shows segment-level completion across the full backlog.


In [8]:
if RUN_STAGE_2_ASR:
    asr_summary = run_asr_stage(CFG)
    print(asr_summary)
else:
    print("Skipped Stage 2 (RUN_STAGE_2_ASR=False)")


ASR segments already done: 185/610 (30.3%)


Stage 2 · ASR recognize segments:   0%|          | 0/425 [00:00<?, ?seg/s]

ASR segments completed: 610/610 (100.0%)
{'total_segments': 610, 'processed_segments': 425, 'skipped_segments': 185, 'successful_in_run': 425}


## Stage 3 — Merge full-call transcript

Combines `asr_results.jsonl` into ordered dialogue lines and writes `final_dialogue.json` per call.


In [9]:
import json

if RUN_STAGE_3_MERGE:
    merge_summary = run_merge_stage(CFG)
    print(merge_summary)

    # Preview first available final transcript (skip files like .DS_Store)
    for call_dir in sorted(CFG.segment_output_dir.iterdir()):
        if not call_dir.is_dir() or call_dir.name.startswith("."):
            continue
        dialogue_path = call_dir / "final_dialogue.json"
        if dialogue_path.is_file():
            lines = json.loads(dialogue_path.read_text(encoding="utf-8"))
            print(f"Preview {call_dir.name}:")
            print(json.dumps(lines[:8], ensure_ascii=False, indent=2))
            break
else:
    print("Skipped Stage 3 (RUN_STAGE_3_MERGE=False)")


Stage 3 · Merge transcripts:   0%|          | 0/10 [00:00<?, ?call/s]

Merge transcripts completed: 10/10 (100.0%)
{'total_calls': 10, 'merged_calls': 10}
Preview 00066ab6-b592-4ba3-9117-4a7e5769dc02:
[
  "Agent: à dạ a lô ạ à chị ạ em là hòa trương viên ở bên u c à em thấy mình có bên",
  "Customer: ừ ừ",
  "Agent: lại thông tin luôn là quan tâm đến giải pháp con mất hay ai bên em nên là em gọi lại hộ thế chị ạ",
  "Customer: ừ ừ ừ em",
  "Agent: dạ vâng hiện tại là mình đang làm việc trong lĩnh vực bất động ộng sản đúng không chị",
  "Customer: đúng rồi",
  "Agent: lần này là mình đang muốn chạy gọi để bán dự án mới đúng không hả chị",
  "Customer: đúng rồi"
]
